# Neural Computing Coursework: MLP vs SVM on Credit-Card Fraud Detection

**Author:** `<Author Name>` (`<Student ID>`)

This notebook compares a fully-connected **Multilayer Perceptron** (PyTorch) and a
**Support Vector Machine** (scikit-learn) on the Dal Pozzolo et al. (2015)
credit-card fraud dataset (Zenodo record 7395559). It is organised into 12 sections:

1. Environment & reproducibility
2. Data loading & EDA
3. Train / validation / test split
4. Preprocessing
5. MLP: model, training, hyperparameter search
6. SVM: pipeline, grid search
7. Class-imbalance ablation (none / class weights / SMOTE)
8. Final evaluation with threshold tuning
9. Multi-seed robustness
10. Training-size scaling
11. Error analysis & model disagreement
12. Save best models + reload sanity check

**Budget:** under 10 minutes on CPU. GPU further accelerates the MLP portion.

**Hypothesis.** *On this dataset, a kernel SVM and an MLP will achieve comparable
ranking performance (ROC-AUC), but will differ meaningfully in (a) the
precision–recall trade-off at operating thresholds, (b) sensitivity to class
imbalance handling (no reweighting vs class weights vs SMOTE), and (c)
computational cost as a function of training set size.*

## 1. Environment & reproducibility

We pin all random seeds (`random`, `numpy`, `torch`, `torch.cuda`), force cuDNN
determinism, and detect the device automatically so the notebook runs identically
on a laptop CPU, a SLURM GPU node, or a CPU-only HPC partition.

The `get_cpu_count()` helper respects `SLURM_CPUS_PER_TASK` so scikit-learn's
grid search uses only the cores SLURM actually allocated — not every core on
the node.

The `MPLBACKEND=Agg` guard prevents matplotlib from trying to connect to an X
server on headless HPC nodes (a common silent-fail).

In [1]:
# Core scientific stack
import os
import sys
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

# Headless-safe backend for SLURM / remote shells without X forwarding.
if not os.environ.get('DISPLAY') and os.environ.get('MPLBACKEND') != 'Agg':
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, precision_recall_curve,
    roc_curve, confusion_matrix,
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Persistence
import joblib

# Silence sklearn's harmless InconsistentVersionWarning when reloading artefacts
# saved on a different sklearn patch version (HPC vs local).
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

# -----------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Why: cuDNN's default algorithm auto-selection is nondeterministic. Setting
# deterministic=True and benchmark=False trades a small speed loss for
# bitwise-reproducible forward/backward passes across runs on the same GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    """Return CUDA device if available, else CPU."""
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def get_cpu_count() -> int:
    """Respect SLURM allocation; fall back to os.cpu_count().

    Using ``n_jobs=-1`` on SLURM grabs every core on the compute node — not
    just the cores SLURM actually allocated to your job — and invites the
    scheduler to kill you. Always read ``SLURM_CPUS_PER_TASK`` first.
    """
    return int(os.environ.get('SLURM_CPUS_PER_TASK', os.cpu_count() or 1))


DEVICE = get_device()
N_JOBS = get_cpu_count()

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
ROOT = Path.cwd()
DATA_PATH = ROOT / 'creditcard.csv'
REDUCED_PATH = ROOT / 'creditcard_reduced.csv'
OUT_DIR = ROOT / 'outputs'
MODEL_DIR = ROOT / 'models'
OUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

# -----------------------------------------------------------------------------
# Plot style
# -----------------------------------------------------------------------------
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

print(f"Python       : {sys.version.split()[0]}")
print(f"NumPy        : {np.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"torch        : {torch.__version__}")
print(f"Device       : {DEVICE}")
print(f"Parallel jobs: {N_JOBS}")
print(f"Seed         : {SEED}")
print(f"Outputs dir  : {OUT_DIR}")
print(f"Models dir   : {MODEL_DIR}")

Python       : 3.11.15
NumPy        : 1.26.4
pandas       : 2.2.2
torch        : 2.2.2+cu121
Device       : cpu
Parallel jobs: 16
Seed         : 42
Outputs dir  : /users/adhp492/neuralComputingHPC/outputs
Models dir   : /users/adhp492/neuralComputingHPC/models


## 2. Data loading & EDA

On first run, we download the full 284,807-row dataset from Zenodo
(Dal Pozzolo et al. 2015), then build a stratified subsample of **all 492
fraud cases plus 10,000 random legitimate transactions** (10,492 rows, 4.69%
positive class). This keeps every minority-class example while shrinking the
training-time budget so kernel SVMs — whose cost is O(n²)–O(n³) — remain
tractable, and enables proper CV + multi-seed runs under a 10-minute budget.

The subsampling is itself a caveat and is discussed in the paper's Limitations
section.

In [2]:
# Download the full dataset if missing. Uses the standalone helper in scripts/.
if not DATA_PATH.exists():
    print(f"{DATA_PATH.name} not found — downloading (~150 MB)...")
    # Use the helper as a module to avoid re-implementing the download logic.
    import subprocess
    result = subprocess.run(
        [sys.executable, str(ROOT / 'scripts' / 'download_data.py'),
         '--out', str(DATA_PATH)],
        check=True,
    )
else:
    print(f"{DATA_PATH.name} already present ({DATA_PATH.stat().st_size/1e6:.1f} MB).")

creditcard.csv already present (150.8 MB).


In [3]:
# Build the stratified subsample deterministically (seed=42).
if not REDUCED_PATH.exists():
    print("Building creditcard_reduced.csv (this is a one-off step)...")
    df_full = pd.read_csv(DATA_PATH)
    fraud = df_full[df_full['Class'] == 1]
    legit = df_full[df_full['Class'] == 0].sample(n=10_000, random_state=SEED)
    df = pd.concat([fraud, legit], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    df.to_csv(REDUCED_PATH, index=False)
    print(f"Wrote {REDUCED_PATH.name}: {len(df)} rows, {df['Class'].sum()} frauds ({df['Class'].mean()*100:.2f}%).")
    del df_full
else:
    print(f"{REDUCED_PATH.name} already present.")

df = pd.read_csv(REDUCED_PATH)
print(f"\nLoaded: shape={df.shape}, missing={df.isna().sum().sum()}")
print(f"Class distribution: {df['Class'].value_counts().to_dict()}")
print(f"Fraud rate: {df['Class'].mean()*100:.3f}%")

creditcard_reduced.csv already present.

Loaded: shape=(10492, 31), missing=0
Class distribution: {0: 10000, 1: 492}
Fraud rate: 4.689%


In [4]:
# Summary statistics for Time, Amount, Class.
display(df[['Time', 'Amount', 'Class']].describe())

,Time,Amount,Class
count,10492.000000,10492.000000,10492.000000
mean,94584.647446,88.045345,0.046893
std,47483.849461,223.107826,0.211419
min,0.000000,0.000000,0.000000
25%,54186.000000,4.990000,0.000000
50%,85014.000000,21.130000,0.000000
75%,139104.000000,78.372500,0.000000
max,172768.000000,5627.060000,1.000000


In [5]:
# Figure 1: class balance — two panels (count + Amount by class).
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Class'].value_counts().sort_index()
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], counts.values,
            color=['#4C72B0', '#C44E52'])
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Class balance (fraud rate = {df["Class"].mean()*100:.2f}%)')

for cls, colour, label in [(0, '#4C72B0', 'Legitimate'), (1, '#C44E52', 'Fraud')]:
    subset = df[df['Class'] == cls]['Amount']
    axes[1].hist(subset[subset > 0], bins=40, alpha=0.6, label=label,
                 color=colour, density=True)
axes[1].set_xscale('log')
axes[1].set_xlabel('Transaction amount (log scale)')
axes[1].set_ylabel('Density')
axes[1].set_title('Amount distribution by class')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig01_class_balance.png')
plt.show()

In [6]:
# Top 10 features by absolute correlation with Class — quick sanity check.
feature_cols = [c for c in df.columns if c != 'Class']
corr = df[feature_cols + ['Class']].corr()['Class'].drop('Class')
top10 = corr.abs().sort_values(ascending=False).head(10)
print("Top 10 features by |corr| with Class:")
for name, v in top10.items():
    sign = corr[name]
    print(f"  {name:<8s}  |corr| = {v:.4f}  (signed: {sign:+.4f})")

Top 10 features by |corr| with Class:
  V14       |corr| = 0.7527  (signed: -0.7527)
  V12       |corr| = 0.6984  (signed: -0.6984)
  V17       |corr| = 0.6454  (signed: -0.6454)
  V10       |corr| = 0.6307  (signed: -0.6307)
  V16       |corr| = 0.5993  (signed: -0.5993)
  V3        |corr| = 0.5806  (signed: -0.5806)
  V11       |corr| = 0.5786  (signed: +0.5786)
  V4        |corr| = 0.5385  (signed: +0.5385)
  V7        |corr| = 0.5259  (signed: -0.5259)
  V18       |corr| = 0.4215  (signed: -0.4215)


## 3. Train / validation / test split

Three-way stratified split: **60% train / 20% validation / 20% test**. We use
`train_test_split` twice, stratifying on `y` each time and pinning
`random_state=SEED` so both the main run and `test.ipynb` reconstruct the same
test set.

In [7]:
X = df[feature_cols].values
y = df['Class'].values

# First split: 60% train+val-combined is produced from 80% train+val, 20% test.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED,
)
# Second split: of the 80% that isn't test, take 75% for train and 25% for val,
# giving 60/20/20 overall.
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=SEED,
)

for name, arr in [('train', y_train), ('val', y_val), ('test', y_test), ('trainval', y_trainval)]:
    print(f"  {name:<8s}  n={len(arr):>5d}  frauds={int(arr.sum()):>3d}  fraud_rate={arr.mean()*100:.3f}%")

  train     n= 6294  frauds=295  fraud_rate=4.687%
  val       n= 2099  frauds= 99  fraud_rate=4.717%
  test      n= 2099  frauds= 98  fraud_rate=4.669%
  trainval  n= 8393  frauds=394  fraud_rate=4.694%


## 4. Preprocessing

Both methods benefit from standardised inputs: MLPs because gradient magnitudes
are scale-sensitive, SVMs because the RBF kernel width is scale-dependent.
We fit the `StandardScaler` **on training data only** to prevent information
from val / test leaking into the preprocessing stage. The SVM pipeline has its
own scaler nested inside so `GridSearchCV` can re-fit it per fold — both
are honest, neither leaks.

In [8]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

# Sanity check: training features are zero-mean, unit-std by construction.
print("After scaling (first 3 features, train split):")
print(f"  means: {X_train_s[:, :3].mean(axis=0)}")
print(f"  stds : {X_train_s[:, :3].std(axis=0)}")

After scaling (first 3 features, train split):
  means: [-6.40309752e-17 -4.43806980e-17 -9.88511254e-17]
  stds : [1. 1. 1.]


## 5. MLP: model, training, hyperparameter search

### 5.1 Model definition

A configurable feed-forward network with ReLU activations and dropout
regularisation between hidden layers. A single logit output is fed to
`BCEWithLogitsLoss`, which is numerically more stable than applying sigmoid
followed by `BCELoss`.

In [9]:
class MLP(nn.Module):
    """Fully-connected binary classifier with configurable depth / dropout.

    Parameters
    ----------
    in_dim : int
        Number of input features.
    hidden_sizes : tuple[int, ...]
        Width of each hidden layer, in order.
    dropout : float
        Dropout probability applied after each hidden layer's ReLU.

    The forward pass returns a single logit per example (shape ``(N,)``).
    """

    def __init__(self, in_dim: int, hidden_sizes=(64, 32), dropout: float = 0.2):
        super().__init__()
        layers: list[nn.Module] = []
        prev = in_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)

    @torch.no_grad()
    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        """Sigmoid of logits, for PR/ROC curves at inference time."""
        self.eval()
        return torch.sigmoid(self(x))

### 5.2 Training loop with early stopping on validation PR-AUC

We stop on PR-AUC rather than loss: validation loss can stay low while the
model silently ignores the minority class (at 4.69% fraud, predicting "legit"
for everything is ~95% accurate). `pos_weight` in `BCEWithLogitsLoss` supports
the class-weighted variant used in Section 7.

In [10]:
def train_mlp(
    X_tr, y_tr, X_vl, y_vl,
    hidden_sizes=(64, 32), dropout=0.2, lr=1e-3, weight_decay=1e-4,
    epochs=40, batch_size=256, pos_weight=None, patience=8, verbose=False,
):
    """Train an MLP with early stopping on validation PR-AUC.

    Parameters
    ----------
    X_tr, y_tr : np.ndarray
        Training features + labels (already scaled).
    X_vl, y_vl : np.ndarray
        Validation features + labels (already scaled).
    hidden_sizes : tuple[int, ...]
        Hidden layer widths.
    dropout : float
        Dropout probability.
    lr : float
        Adam learning rate.
    weight_decay : float
        Adam L2 regularisation coefficient.
    epochs : int
        Upper bound on training epochs.
    batch_size : int
        Mini-batch size.
    pos_weight : float or None
        Positive-class weight in BCEWithLogitsLoss; ``None`` disables weighting.
    patience : int
        Stop if validation PR-AUC fails to improve for this many epochs.
    verbose : bool
        Print per-epoch metrics.

    Returns
    -------
    dict
        ``model``, ``history`` (per-epoch loss + val PR-AUC + val ROC-AUC),
        ``best_val_pr_auc``, ``best_epoch``, ``epochs_trained``.
    """
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32, device=DEVICE)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32, device=DEVICE)
    X_vl_t = torch.tensor(X_vl, dtype=torch.float32, device=DEVICE)

    model = MLP(X_tr.shape[1], hidden_sizes=hidden_sizes, dropout=dropout).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    if pos_weight is not None:
        pw = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        criterion = nn.BCEWithLogitsLoss()

    loader = DataLoader(
        TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=True,
    )

    history = {'loss': [], 'val_pr_auc': [], 'val_roc_auc': []}
    best_pr, best_state, best_epoch, bad_epochs = -1.0, None, 0, 0

    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * xb.size(0)
        epoch_loss = total / len(loader.dataset)

        proba = model.predict_proba(X_vl_t).cpu().numpy()
        pr = average_precision_score(y_vl, proba)
        roc = roc_auc_score(y_vl, proba)
        history['loss'].append(epoch_loss)
        history['val_pr_auc'].append(pr)
        history['val_roc_auc'].append(roc)

        if pr > best_pr + 1e-6:
            best_pr, best_epoch, bad_epochs = pr, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1

        if verbose:
            print(f"  epoch {epoch:3d}  loss={epoch_loss:.4f}  val PR-AUC={pr:.4f}  val ROC-AUC={roc:.4f}"
                  f"  (best epoch {best_epoch})")

        if bad_epochs >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return {
        'model': model,
        'history': history,
        'best_val_pr_auc': best_pr,
        'best_epoch': best_epoch,
        'epochs_trained': len(history['loss']),
    }

### 5.3 Baseline run + learning curves

A single sanity-check training run with sensible defaults. We save the learning
curve figure so the paper and supplementary can reference it.

In [11]:
t0 = time.time()
baseline = train_mlp(
    X_train_s, y_train, X_val_s, y_val,
    hidden_sizes=(64, 32), dropout=0.2, lr=1e-3,
    epochs=40, patience=8, verbose=False,
)
t_baseline = time.time() - t0
print(f"Baseline MLP: best val PR-AUC = {baseline['best_val_pr_auc']:.4f} "
      f"at epoch {baseline['best_epoch']}/{baseline['epochs_trained']} "
      f"({t_baseline:.1f}s)")

# Figure 2: learning curves.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(baseline['history']['loss'], label='train loss', color='#4C72B0')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE loss')
axes[0].set_title('Training loss')
axes[0].legend()

axes[1].plot(baseline['history']['val_pr_auc'], label='val PR-AUC', color='#C44E52')
axes[1].plot(baseline['history']['val_roc_auc'], label='val ROC-AUC', color='#55A868')
axes[1].axvline(baseline['best_epoch'] - 1, ls='--', color='grey',
                label=f'best epoch ({baseline["best_epoch"]})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_title('Validation metrics')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig02_mlp_learning_curves.png')
plt.show()

Baseline MLP: best val PR-AUC = 0.9349 at epoch 34/40 (4.6s)


### 5.4 Hyperparameter search — 5-fold stratified CV

We match the SVM's `GridSearchCV` methodology exactly: 5 stratified folds,
scored by mean PR-AUC, with a per-fold `StandardScaler` fitted inside the fold
so there is no leakage through the preprocessing step.

**Grid:** hidden_sizes × dropout × lr = 5 × 3 × 3 = 45 configurations.
Each is trained for up to 25 epochs with early stopping (patience=6) per fold
to keep the total budget manageable.

In [12]:
def cv_score_mlp(X, y, hidden_sizes, dropout, lr, n_splits=5,
                 epochs=25, patience=6, batch_size=256):
    """5-fold stratified CV score (mean + std PR-AUC) for one MLP config.

    Each fold fits its own StandardScaler on the fold's training indices to
    avoid leakage across folds.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_prs = []
    for tr_idx, vl_idx in skf.split(X, y):
        sc = StandardScaler().fit(X[tr_idx])
        Xt, Xv = sc.transform(X[tr_idx]), sc.transform(X[vl_idx])
        res = train_mlp(
            Xt, y[tr_idx], Xv, y[vl_idx],
            hidden_sizes=hidden_sizes, dropout=dropout, lr=lr,
            epochs=epochs, patience=patience, batch_size=batch_size,
        )
        fold_prs.append(res['best_val_pr_auc'])
    return float(np.mean(fold_prs)), float(np.std(fold_prs)), fold_prs

In [13]:
# Search grid — identical structure to the SVM's grid (kernel/C/gamma/degree
# is the SVM analogue of hidden_sizes/dropout/lr).
mlp_grid = []
for hs in [(32,), (64,), (64, 32), (128, 64), (128, 64, 32)]:
    for d in [0.0, 0.2, 0.3]:
        for lr in [1e-4, 1e-3, 3e-3]:
            mlp_grid.append({'hidden_sizes': hs, 'dropout': d, 'lr': lr})

print(f"MLP grid size: {len(mlp_grid)} configurations")
print("Running 5-fold CV over the grid (this is the main MLP training cost)...")

t0 = time.time()
mlp_results = []
# Use X_trainval / y_trainval (combined 80%) so the search scope matches the
# SVM's: both models hold out the same 20% test set.
for i, cfg in enumerate(mlp_grid, 1):
    mean_pr, std_pr, folds = cv_score_mlp(
        X_trainval, y_trainval,
        hidden_sizes=cfg['hidden_sizes'], dropout=cfg['dropout'], lr=cfg['lr'],
    )
    mlp_results.append({**cfg, 'mean_pr_auc': mean_pr, 'std_pr_auc': std_pr, 'fold_pr_aucs': folds})
    print(f"  [{i:2d}/{len(mlp_grid)}] hs={cfg['hidden_sizes']} d={cfg['dropout']} "
          f"lr={cfg['lr']:.0e}  CV PR-AUC = {mean_pr:.4f} ± {std_pr:.4f}")
t_mlp_search = time.time() - t0
print(f"\nMLP grid search complete in {t_mlp_search:.1f}s.")

mlp_df = pd.DataFrame(mlp_results).sort_values('mean_pr_auc', ascending=False).reset_index(drop=True)
print("\nTop 5 MLP configs:")
display(mlp_df.head(5)[['hidden_sizes', 'dropout', 'lr', 'mean_pr_auc', 'std_pr_auc']])

MLP grid size: 45 configurations
Running 5-fold CV over the grid (this is the main MLP training cost)...


  [ 1/45] hs=(32,) d=0.0 lr=1e-04  CV PR-AUC = 0.8428 ± 0.0555


  [ 2/45] hs=(32,) d=0.0 lr=1e-03  CV PR-AUC = 0.9194 ± 0.0284


  [ 3/45] hs=(32,) d=0.0 lr=3e-03  CV PR-AUC = 0.9269 ± 0.0241


  [ 4/45] hs=(32,) d=0.2 lr=1e-04  CV PR-AUC = 0.8417 ± 0.0551


...

  [42/45] hs=(128, 64, 32) d=0.2 lr=3e-03  CV PR-AUC = 0.9281 ± 0.0268


  [43/45] hs=(128, 64, 32) d=0.3 lr=1e-04  CV PR-AUC = 0.9089 ± 0.0431


  [44/45] hs=(128, 64, 32) d=0.3 lr=1e-03  CV PR-AUC = 0.9275 ± 0.0280


  [45/45] hs=(128, 64, 32) d=0.3 lr=3e-03  CV PR-AUC = 0.9274 ± 0.0259

MLP grid search complete in 408.3s.

Top 5 MLP configs:


,hidden_sizes,dropout,lr,mean_pr_auc,std_pr_auc
0,"(128, 64)",0.2,0.001,0.930953,0.026607
1,"(128, 64)",0.0,0.001,0.930513,0.026924
2,"(128, 64)",0.3,0.001,0.930484,0.026790
3,"(128, 64)",0.0,0.003,0.930429,0.024767
4,"(128, 64)",0.2,0.003,0.930197,0.026894


### 5.5 Retrain best MLP config on X_train

We retrain on the 60% `X_train` split (not the combined 80%) so the 20%
validation split stays clean for threshold tuning in Section 8.

In [14]:
best_mlp_cfg = mlp_df.iloc[0].to_dict()
print(f"Best MLP: hidden_sizes={best_mlp_cfg['hidden_sizes']}  "
      f"dropout={best_mlp_cfg['dropout']}  lr={best_mlp_cfg['lr']}  "
      f"(CV PR-AUC {best_mlp_cfg['mean_pr_auc']:.4f})")

mlp_fit = train_mlp(
    X_train_s, y_train, X_val_s, y_val,
    hidden_sizes=best_mlp_cfg['hidden_sizes'],
    dropout=best_mlp_cfg['dropout'],
    lr=best_mlp_cfg['lr'],
    epochs=40, patience=8,
)
print(f"Validation PR-AUC (refit): {mlp_fit['best_val_pr_auc']:.4f} "
      f"(epoch {mlp_fit['best_epoch']}/{mlp_fit['epochs_trained']})")

Best MLP: hidden_sizes=(128, 64)  dropout=0.2  lr=0.001  (CV PR-AUC 0.9310)


Validation PR-AUC (refit): 0.9432 (epoch 27/35)


## 6. SVM: pipeline and grid search

The pipeline nests a `StandardScaler` inside the SVC so `GridSearchCV` can refit
the scaler on each CV fold's training subset — preventing leakage without us
having to hand-code per-fold preprocessing.

`probability=True` enables Platt scaling (Platt 1999), which fits a small
logistic calibrator on top of the SVM decision function. This is slower but
required because PR/ROC curves need probability-like scores.

We fit on `X_trainval` so the search scope matches the MLP's.

In [15]:
svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(probability=True, random_state=SEED)),
])

svm_grid = [
    {'svc__kernel': ['linear'], 'svc__C': [0.1, 1, 10]},
    {'svc__kernel': ['rbf'],    'svc__C': [0.1, 1, 10], 'svc__gamma': ['scale', 0.01, 0.1]},
    {'svc__kernel': ['poly'],   'svc__C': [0.1, 1, 10], 'svc__gamma': ['scale', 0.01, 0.1],
     'svc__degree': [2, 3]},
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
svm_search = GridSearchCV(
    svm_pipe, svm_grid, cv=cv, scoring='average_precision',
    n_jobs=N_JOBS, refit=True, verbose=0,
)

t0 = time.time()
svm_search.fit(X_trainval, y_trainval)
t_svm_search = time.time() - t0
print(f"SVM grid search complete in {t_svm_search:.1f}s "
      f"(n_jobs={N_JOBS}, {len(svm_search.cv_results_['params'])} configs).")
print(f"Best params: {svm_search.best_params_}")
print(f"Best CV PR-AUC: {svm_search.best_score_:.4f}")

SVM grid search complete in 38.5s (n_jobs=16, 30 configs).
Best params: {'svc__C': 0.1, 'svc__kernel': 'linear'}
Best CV PR-AUC: 0.9240


In [16]:
# Full CV results table (goes into the supplementary).
svm_cv_df = pd.DataFrame(svm_search.cv_results_)[
    ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
].sort_values('rank_test_score').reset_index(drop=True)
svm_cv_df.columns = ['params', 'cv_mean_pr_auc', 'cv_std_pr_auc', 'rank']
print("Top 5 SVM configs:")
display(svm_cv_df.head(5))

Top 5 SVM configs:


,params,cv_mean_pr_auc,cv_std_pr_auc,rank
0,"{'svc__C': 0.1, 'svc__kernel': 'linear'}",0.923987,0.026664,1
1,"{'svc__C': 10, 'svc__gamma': 0.01, 'svc__kerne...",0.921967,0.023427,2
2,"{'svc__C': 10, 'svc__kernel': 'linear'}",0.921396,0.028547,3
3,"{'svc__C': 1, 'svc__kernel': 'linear'}",0.920882,0.027942,4
4,"{'svc__C': 10, 'svc__gamma': 'scale', 'svc__ke...",0.914313,0.026596,5


In [17]:
# Refit best SVM on X_train only (matching MLP's scope) for fair downstream eval.
best_svm_params = svm_search.best_params_
svm_refit = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(probability=True, random_state=SEED,
                **{k.replace('svc__', ''): v for k, v in best_svm_params.items()})),
])
svm_refit.fit(X_train, y_train)

svm_val_proba = svm_refit.predict_proba(X_val)[:, 1]
svm_val_pr = average_precision_score(y_val, svm_val_proba)
print(f"SVM val PR-AUC (refit on X_train): {svm_val_pr:.4f}")

SVM val PR-AUC (refit on X_train): 0.9329


## 7. Class-imbalance ablation

Three strategies × two models = six cells:

| Strategy | MLP | SVM |
|---|---|---|
| None | `BCEWithLogitsLoss` no weights | `class_weight=None` |
| Class weights | `pos_weight = n_neg / n_pos` | `class_weight='balanced'` |
| SMOTE | oversample **training only** | `imblearn.Pipeline` nests SMOTE inside each CV fold |

**Leakage guard:** SMOTE must never touch validation or test data. For the MLP
we apply it once to `X_train`. For the SVM we use `imblearn.Pipeline` so SMOTE
runs inside each fold during the ablation re-fit.

In [18]:
pos_weight_val = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print(f"Class-weight ratio (n_neg/n_pos on train): {pos_weight_val:.2f}")

def eval_proba(y_true, proba):
    return {
        'pr_auc': average_precision_score(y_true, proba),
        'roc_auc': roc_auc_score(y_true, proba),
    }

ablation_rows = []

# --- MLP variants -----------------------------------------------------------
# 1. None
r = train_mlp(X_train_s, y_train, X_val_s, y_val,
              hidden_sizes=best_mlp_cfg['hidden_sizes'],
              dropout=best_mlp_cfg['dropout'], lr=best_mlp_cfg['lr'])
ablation_rows.append({'model': 'MLP', 'strategy': 'none',
                      **eval_proba(y_val, r['model'].predict_proba(
                          torch.tensor(X_val_s, dtype=torch.float32, device=DEVICE)
                      ).cpu().numpy())})

# 2. Class weights
r = train_mlp(X_train_s, y_train, X_val_s, y_val,
              hidden_sizes=best_mlp_cfg['hidden_sizes'],
              dropout=best_mlp_cfg['dropout'], lr=best_mlp_cfg['lr'],
              pos_weight=pos_weight_val)
ablation_rows.append({'model': 'MLP', 'strategy': 'class_weight',
                      **eval_proba(y_val, r['model'].predict_proba(
                          torch.tensor(X_val_s, dtype=torch.float32, device=DEVICE)
                      ).cpu().numpy())})

# 3. SMOTE (on training only, before scaling stays valid because SMOTE uses
# Euclidean distance — we operate in the already-scaled space).
smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train_s, y_train)
r = train_mlp(X_train_sm, y_train_sm, X_val_s, y_val,
              hidden_sizes=best_mlp_cfg['hidden_sizes'],
              dropout=best_mlp_cfg['dropout'], lr=best_mlp_cfg['lr'])
ablation_rows.append({'model': 'MLP', 'strategy': 'smote',
                      **eval_proba(y_val, r['model'].predict_proba(
                          torch.tensor(X_val_s, dtype=torch.float32, device=DEVICE)
                      ).cpu().numpy())})

# --- SVM variants -----------------------------------------------------------
svm_params = {k.replace('svc__', ''): v for k, v in best_svm_params.items()}

# 1. None
s = Pipeline([('scaler', StandardScaler()),
              ('svc', SVC(probability=True, random_state=SEED, **svm_params))]).fit(X_train, y_train)
ablation_rows.append({'model': 'SVM', 'strategy': 'none',
                      **eval_proba(y_val, s.predict_proba(X_val)[:, 1])})

# 2. class_weight='balanced'
s = Pipeline([('scaler', StandardScaler()),
              ('svc', SVC(probability=True, random_state=SEED,
                          class_weight='balanced', **svm_params))]).fit(X_train, y_train)
ablation_rows.append({'model': 'SVM', 'strategy': 'class_weight',
                      **eval_proba(y_val, s.predict_proba(X_val)[:, 1])})

# 3. SMOTE (via imblearn Pipeline — SMOTE only on training data inside the pipe).
s = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=SEED)),
    ('svc', SVC(probability=True, random_state=SEED, **svm_params)),
]).fit(X_train, y_train)
ablation_rows.append({'model': 'SVM', 'strategy': 'smote',
                      **eval_proba(y_val, s.predict_proba(X_val)[:, 1])})

ablation_df = pd.DataFrame(ablation_rows)
print("Class-imbalance ablation (validation PR-AUC / ROC-AUC):")
display(ablation_df.pivot(index='strategy', columns='model', values='pr_auc'))

Class-weight ratio (n_neg/n_pos on train): 20.34


Class-imbalance ablation (validation PR-AUC / ROC-AUC):


model,MLP,SVM
strategy,,
class_weight,0.940017,0.917863
none,0.943219,0.932907
smote,0.934727,0.921220


In [19]:
# Figure 3: grouped bar chart of val PR-AUC by strategy × model.
fig, ax = plt.subplots(figsize=(8, 4.5))
pivot = ablation_df.pivot(index='strategy', columns='model', values='pr_auc')
pivot.plot(kind='bar', ax=ax, color=['#4C72B0', '#C44E52'], rot=0)
ax.set_ylabel('Validation PR-AUC')
ax.set_title('Imbalance-handling strategies')
ax.set_ylim(0, 1.0)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig03_imbalance_ablation.png')
plt.show()

best_strategy = {
    m: ablation_df[ablation_df['model'] == m].sort_values('pr_auc', ascending=False).iloc[0]['strategy']
    for m in ['MLP', 'SVM']
}
print(f"Best strategy per model (by val PR-AUC): {best_strategy}")

Best strategy per model (by val PR-AUC): {'MLP': 'none', 'SVM': 'none'}


## 8. Final evaluation on held-out test set

### 8.1 Threshold tuning on the validation set

The default 0.5 threshold is rarely optimal on imbalanced problems. We scan
200 thresholds on the **validation** set (never test) and pick the one that
maximises F1. This single extra hyperparameter is then reported honestly
alongside the 0.5-default results.

In [20]:
def tune_threshold(y_true, proba, n=200):
    """Return the threshold in (0.01, 0.99) that maximises F1."""
    thresholds = np.linspace(0.01, 0.99, n)
    f1s = np.array([f1_score(y_true, (proba >= t).astype(int), zero_division=0)
                    for t in thresholds])
    best = thresholds[int(f1s.argmax())]
    return float(best), float(f1s.max()), thresholds, f1s

# Refit both models using the best imbalance strategy identified in §7.
def fit_best_mlp(strategy):
    if strategy == 'smote':
        Xt, yt = SMOTE(random_state=SEED).fit_resample(X_train_s, y_train)
        pw = None
    elif strategy == 'class_weight':
        Xt, yt = X_train_s, y_train
        pw = pos_weight_val
    else:
        Xt, yt = X_train_s, y_train
        pw = None
    return train_mlp(Xt, yt, X_val_s, y_val,
                     hidden_sizes=best_mlp_cfg['hidden_sizes'],
                     dropout=best_mlp_cfg['dropout'], lr=best_mlp_cfg['lr'],
                     pos_weight=pw, epochs=40, patience=8)

def fit_best_svm(strategy):
    if strategy == 'smote':
        pipe = ImbPipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=SEED)),
            ('svc', SVC(probability=True, random_state=SEED, **svm_params)),
        ])
    elif strategy == 'class_weight':
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('svc', SVC(probability=True, random_state=SEED,
                        class_weight='balanced', **svm_params)),
        ])
    else:
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('svc', SVC(probability=True, random_state=SEED, **svm_params)),
        ])
    return pipe.fit(X_train, y_train)

final_mlp = fit_best_mlp(best_strategy['MLP'])
final_svm = fit_best_svm(best_strategy['SVM'])

mlp_val_proba = final_mlp['model'].predict_proba(
    torch.tensor(X_val_s, dtype=torch.float32, device=DEVICE)
).cpu().numpy()
svm_val_proba = final_svm.predict_proba(X_val)[:, 1]

mlp_thr, mlp_valf1, mlp_thrs, mlp_f1s = tune_threshold(y_val, mlp_val_proba)
svm_thr, svm_valf1, svm_thrs, svm_f1s = tune_threshold(y_val, svm_val_proba)
print(f"Tuned thresholds: MLP={mlp_thr:.3f} (val F1={mlp_valf1:.4f})   "
      f"SVM={svm_thr:.3f} (val F1={svm_valf1:.4f})")

Tuned thresholds: MLP=0.168 (val F1=0.9167)   SVM=0.010 (val F1=0.8901)


In [21]:
# Figure 4: F1 vs threshold for both models.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(mlp_thrs, mlp_f1s, label='MLP', color='#4C72B0')
ax.plot(svm_thrs, svm_f1s, label='SVM', color='#C44E52')
ax.axvline(mlp_thr, ls='--', color='#4C72B0', alpha=0.7, label=f'MLP tuned ({mlp_thr:.2f})')
ax.axvline(svm_thr, ls='--', color='#C44E52', alpha=0.7, label=f'SVM tuned ({svm_thr:.2f})')
ax.axvline(0.5, ls=':', color='grey', label='default (0.50)')
ax.set_xlabel('Threshold')
ax.set_ylabel('Validation F1')
ax.set_title('F1 vs threshold')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig04_threshold_tuning.png')
plt.show()

### 8.2 Test-set results at default and tuned thresholds

In [22]:
def evaluate(y_true, proba, threshold=0.5):
    """Return ROC-AUC, PR-AUC, F1/precision/recall at ``threshold``."""
    pred = (proba >= threshold).astype(int)
    return {
        'roc_auc': roc_auc_score(y_true, proba),
        'pr_auc': average_precision_score(y_true, proba),
        'f1': f1_score(y_true, pred, zero_division=0),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
    }

mlp_test_proba = final_mlp['model'].predict_proba(
    torch.tensor(X_test_s, dtype=torch.float32, device=DEVICE)
).cpu().numpy()
svm_test_proba = final_svm.predict_proba(X_test)[:, 1]

test_rows = [
    {'model': 'MLP', 'threshold': 'default 0.50', **evaluate(y_test, mlp_test_proba, 0.5)},
    {'model': 'MLP', 'threshold': f'tuned {mlp_thr:.3f}', **evaluate(y_test, mlp_test_proba, mlp_thr)},
    {'model': 'SVM', 'threshold': 'default 0.50', **evaluate(y_test, svm_test_proba, 0.5)},
    {'model': 'SVM', 'threshold': f'tuned {svm_thr:.3f}', **evaluate(y_test, svm_test_proba, svm_thr)},
]
test_df = pd.DataFrame(test_rows)
print("Test-set metrics:")
display(test_df.round(4))

Test-set metrics:


,model,threshold,roc_auc,pr_auc,f1,precision,recall
0,MLP,default 0.50,0.9826,0.9149,0.8715,0.9630,0.7959
1,MLP,tuned 0.168,0.9826,0.9149,0.8796,0.9032,0.8571
2,SVM,default 0.50,0.9799,0.9079,0.8333,1.0000,0.7143
3,SVM,tuned 0.010,0.9799,0.9079,0.8235,0.9722,0.7143


In [23]:
# Figure 5: ROC + PR curves on the test set.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for proba, label, colour in [
    (mlp_test_proba, 'MLP', '#4C72B0'),
    (svm_test_proba, 'SVM', '#C44E52'),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{label} (ROC-AUC={roc:.3f})', color=colour)

    prec, rec, _ = precision_recall_curve(y_test, proba)
    pr = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, label=f'{label} (PR-AUC={pr:.3f})', color=colour)

axes[0].plot([0, 1], [0, 1], '--', color='grey', lw=1)
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].set_title('ROC curves — test set')
axes[0].legend(loc='lower right')

axes[1].axhline(y_test.mean(), ls='--', color='grey', lw=1, label=f'baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-recall curves — test set')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig05_test_curves.png')
plt.show()

## 9. Multi-seed robustness

A single seed says almost nothing about how noisy these estimates are. We
re-run the full pipeline (split → scale → train → tune threshold → test) for
five seeds and report mean ± std.

In [24]:
def run_seed(seed):
    """Re-run the full pipeline end-to-end for a given seed; return test metrics."""
    X_tv, X_te, y_tv, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)
    X_tr, X_vl, y_tr, y_vl = train_test_split(X_tv, y_tv, test_size=0.25, stratify=y_tv, random_state=seed)

    sc = StandardScaler().fit(X_tr)
    X_tr_s, X_vl_s, X_te_s = sc.transform(X_tr), sc.transform(X_vl), sc.transform(X_te)

    # MLP
    res = train_mlp(X_tr_s, y_tr, X_vl_s, y_vl,
                    hidden_sizes=best_mlp_cfg['hidden_sizes'],
                    dropout=best_mlp_cfg['dropout'], lr=best_mlp_cfg['lr'],
                    epochs=40, patience=8,
                    pos_weight=pos_weight_val if best_strategy['MLP'] == 'class_weight' else None)
    mlp_vl = res['model'].predict_proba(torch.tensor(X_vl_s, dtype=torch.float32, device=DEVICE)).cpu().numpy()
    mlp_te = res['model'].predict_proba(torch.tensor(X_te_s, dtype=torch.float32, device=DEVICE)).cpu().numpy()
    mlp_t, _, _, _ = tune_threshold(y_vl, mlp_vl)

    # SVM (no SMOTE here to keep it fast — we've already validated SMOTE in §7)
    svm_kwargs = {**svm_params}
    if best_strategy['SVM'] == 'class_weight':
        svm_kwargs['class_weight'] = 'balanced'
    svm = Pipeline([('scaler', StandardScaler()),
                    ('svc', SVC(probability=True, random_state=seed, **svm_kwargs))]).fit(X_tr, y_tr)
    svm_vl = svm.predict_proba(X_vl)[:, 1]
    svm_te = svm.predict_proba(X_te)[:, 1]
    svm_t, _, _, _ = tune_threshold(y_vl, svm_vl)

    return {
        'seed': seed,
        'mlp': evaluate(y_te, mlp_te, mlp_t),
        'svm': evaluate(y_te, svm_te, svm_t),
        'thresholds': (mlp_t, svm_t),
    }

seeds = [42, 123, 2024, 7, 31415]
seed_results = []
t0 = time.time()
for s in seeds:
    torch.manual_seed(s); np.random.seed(s); random.seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    r = run_seed(s)
    seed_results.append(r)
    print(f"  seed={s:>5d}  MLP PR-AUC={r['mlp']['pr_auc']:.4f}  "
          f"SVM PR-AUC={r['svm']['pr_auc']:.4f}")
print(f"Multi-seed run complete in {time.time()-t0:.1f}s.")

# Aggregate
def agg(results, model):
    df = pd.DataFrame([r[model] for r in results])
    return df.agg(['mean', 'std']).T.rename(columns={'mean': f'{model}_mean', 'std': f'{model}_std'})

summary = pd.concat([agg(seed_results, 'mlp'), agg(seed_results, 'svm')], axis=1)
print("\nAggregate test metrics (mean ± std over 5 seeds):")
display(summary.round(4))

  seed=   42  MLP PR-AUC=0.9149  SVM PR-AUC=0.9079


  seed=  123  MLP PR-AUC=0.9272  SVM PR-AUC=0.9092


  seed= 2024  MLP PR-AUC=0.8829  SVM PR-AUC=0.8768


  seed=    7  MLP PR-AUC=0.9222  SVM PR-AUC=0.9165


  seed=31415  MLP PR-AUC=0.9225  SVM PR-AUC=0.9122
Multi-seed run complete in 18.9s.

Aggregate test metrics (mean ± std over 5 seeds):


,mlp_mean,mlp_std,svm_mean,svm_std
roc_auc,0.9786,0.0096,0.9765,0.0074
pr_auc,0.9139,0.0179,0.9045,0.0159
f1,0.8929,0.0215,0.8694,0.0298
precision,0.9526,0.0381,0.9868,0.0132
recall,0.8408,0.0199,0.7776,0.0417


In [25]:
# Figure 6: per-seed PR-AUC dots with mean bars.
fig, ax = plt.subplots(figsize=(8, 4.5))
mlp_prs = [r['mlp']['pr_auc'] for r in seed_results]
svm_prs = [r['svm']['pr_auc'] for r in seed_results]
ax.scatter([0]*len(mlp_prs), mlp_prs, color='#4C72B0', s=80, alpha=0.7, zorder=3)
ax.scatter([1]*len(svm_prs), svm_prs, color='#C44E52', s=80, alpha=0.7, zorder=3)
ax.bar([0, 1], [np.mean(mlp_prs), np.mean(svm_prs)], alpha=0.25,
       color=['#4C72B0', '#C44E52'],
       yerr=[np.std(mlp_prs), np.std(svm_prs)], capsize=10)
ax.set_xticks([0, 1]); ax.set_xticklabels(['MLP', 'SVM'])
ax.set_ylabel('Test PR-AUC')
ax.set_title('Per-seed test PR-AUC (5 seeds)')
for i, prs in enumerate([mlp_prs, svm_prs]):
    ax.text(i, max(prs)+0.01, f'μ={np.mean(prs):.3f}\nσ={np.std(prs):.3f}',
            ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig06_multi_seed.png')
plt.show()

## 10. Training-size scaling

We train both models at fractions `[0.1, 0.25, 0.5, 0.75, 1.0]` of `X_train`
and record wall-clock training time and test PR-AUC at each. SVM's
O(n²)–O(n³) training cost should show up as a super-linear curve; the MLP
should scale near-linearly with the number of mini-batches.

In [26]:
def stratified_subsample(X, y, frac, seed):
    if frac >= 1.0:
        return X, y
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == 0)[0]
    rng = np.random.default_rng(seed)
    n_pos = max(1, int(len(idx_pos) * frac))
    n_neg = max(1, int(len(idx_neg) * frac))
    sel = np.concatenate([rng.choice(idx_pos, n_pos, replace=False),
                          rng.choice(idx_neg, n_neg, replace=False)])
    rng.shuffle(sel)
    return X[sel], y[sel]

scaling_rows = []
fractions = [0.1, 0.25, 0.5, 0.75, 1.0]
for f in fractions:
    X_sub, y_sub = stratified_subsample(X_train, y_train, f, SEED)
    X_sub_s = scaler.transform(X_sub)

    t0 = time.time()
    res = train_mlp(X_sub_s, y_sub, X_val_s, y_val,
                    hidden_sizes=best_mlp_cfg['hidden_sizes'],
                    dropout=best_mlp_cfg['dropout'], lr=best_mlp_cfg['lr'],
                    epochs=30, patience=6)
    t_mlp = time.time() - t0
    mlp_te = res['model'].predict_proba(torch.tensor(X_test_s, dtype=torch.float32, device=DEVICE)).cpu().numpy()
    mlp_pr = average_precision_score(y_test, mlp_te)

    t0 = time.time()
    svm = Pipeline([('scaler', StandardScaler()),
                    ('svc', SVC(probability=True, random_state=SEED, **svm_params))]).fit(X_sub, y_sub)
    t_svm = time.time() - t0
    svm_te = svm.predict_proba(X_test)[:, 1]
    svm_pr = average_precision_score(y_test, svm_te)

    scaling_rows.append({'fraction': f, 'n_train': len(X_sub),
                         'mlp_time_s': t_mlp, 'mlp_pr_auc': mlp_pr,
                         'svm_time_s': t_svm, 'svm_pr_auc': svm_pr})
    print(f"  frac={f}  n={len(X_sub):>5d}  MLP {t_mlp:6.1f}s / {mlp_pr:.4f}  "
          f"SVM {t_svm:6.1f}s / {svm_pr:.4f}")

scaling_df = pd.DataFrame(scaling_rows)
display(scaling_df.round(4))

  frac=0.1  n=  628  MLP    0.4s / 0.8534  SVM    0.0s / 0.8637


  frac=0.25  n= 1572  MLP    0.6s / 0.8735  SVM    0.0s / 0.8856


  frac=0.5  n= 3146  MLP    1.3s / 0.9101  SVM    0.8s / 0.8496


  frac=0.75  n= 4720  MLP    1.5s / 0.9126  SVM    0.5s / 0.8904


  frac=1.0  n= 6294  MLP    2.5s / 0.9149  SVM    1.2s / 0.9079


,fraction,n_train,mlp_time_s,mlp_pr_auc,svm_time_s,svm_pr_auc
0,0.10,628,0.3912,0.8534,0.0179,0.8637
1,0.25,1572,0.6080,0.8735,0.0363,0.8856
2,0.50,3146,1.2678,0.9101,0.8125,0.8496
3,0.75,4720,1.5312,0.9126,0.4893,0.8904
4,1.00,6294,2.5482,0.9149,1.1610,0.9079


In [27]:
# Figure 7: two-panel — training time (left) + test PR-AUC (right) vs n_train.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(scaling_df['n_train'], scaling_df['mlp_time_s'], 'o-',
             color='#4C72B0', label='MLP')
axes[0].plot(scaling_df['n_train'], scaling_df['svm_time_s'], 's-',
             color='#C44E52', label='SVM')
axes[0].set_xlabel('Training size (n)')
axes[0].set_ylabel('Wall-clock training time (s)')
axes[0].set_title('Training time vs n')
axes[0].legend()

axes[1].plot(scaling_df['n_train'], scaling_df['mlp_pr_auc'], 'o-',
             color='#4C72B0', label='MLP')
axes[1].plot(scaling_df['n_train'], scaling_df['svm_pr_auc'], 's-',
             color='#C44E52', label='SVM')
axes[1].set_xlabel('Training size (n)')
axes[1].set_ylabel('Test PR-AUC')
axes[1].set_title('Test PR-AUC vs n')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig07_training_size_scaling.png')
plt.show()

## 11. Error analysis and model disagreement

Where do MLP and SVM agree, and where do they differ? We look at confusion
matrices at tuned thresholds, an agreement matrix (both correct / both wrong /
disagree), and a probability-space scatter.

In [28]:
mlp_pred = (mlp_test_proba >= mlp_thr).astype(int)
svm_pred = (svm_test_proba >= svm_thr).astype(int)

# Figure 8: confusion matrices at tuned thresholds.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, pred, title in [(axes[0], mlp_pred, f'MLP (thr={mlp_thr:.2f})'),
                         (axes[1], svm_pred, f'SVM (thr={svm_thr:.2f})')]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'],
                cbar=False)
    ax.set_title(title)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig08_confusion_matrices.png')
plt.show()

In [29]:
# Agreement matrix: both correct / both wrong / only MLP right / only SVM right.
mlp_correct = (mlp_pred == y_test)
svm_correct = (svm_pred == y_test)
agree = {
    'both_correct': int((mlp_correct & svm_correct).sum()),
    'both_wrong':   int((~mlp_correct & ~svm_correct).sum()),
    'only_mlp_correct': int((mlp_correct & ~svm_correct).sum()),
    'only_svm_correct': int((~mlp_correct & svm_correct).sum()),
}
total = len(y_test)
print("Agreement matrix on test set:")
for k, v in agree.items():
    print(f"  {k:<20s}  {v:>4d}  ({100*v/total:5.2f}%)")

Agreement matrix on test set:
  both_correct          2061  (98.19%)
  both_wrong              15  ( 0.71%)
  only_mlp_correct        15  ( 0.71%)
  only_svm_correct         8  ( 0.38%)


In [30]:
# Figure 9: probability-space scatter.
fig, ax = plt.subplots(figsize=(7, 5.5))
colours = np.where(y_test == 1, '#C44E52', '#4C72B0')
ax.scatter(svm_test_proba, mlp_test_proba, c=colours, s=15, alpha=0.5)
ax.axvline(svm_thr, ls='--', color='#C44E52', alpha=0.5, label=f'SVM thr {svm_thr:.2f}')
ax.axhline(mlp_thr, ls='--', color='#4C72B0', alpha=0.5, label=f'MLP thr {mlp_thr:.2f}')
ax.set_xlabel('SVM P(fraud)'); ax.set_ylabel('MLP P(fraud)')
ax.set_title('Prediction disagreement — red=fraud, blue=legit')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig09_disagreement.png')
plt.show()

## 12. Save best models

All artefacts persist in `models/`:

- `scaler.joblib` — training-fitted `StandardScaler`
- `best_mlp.pt` — state_dict + architecture + tuned threshold
- `best_svm.joblib` — full sklearn `Pipeline` (scaler + SVC)
- `config.json` — hyperparameters, thresholds, feature columns, expected test metrics

The reload sanity check below is the same logic `test.ipynb` uses to prove
reproducibility without retraining. Budget: **< 10 minutes CPU**, typically
~3–6 minutes locally.

In [31]:
# Save scaler.
joblib.dump(scaler, MODEL_DIR / 'scaler.joblib')

# Save MLP: state_dict + architecture + threshold so we can recreate the module.
torch.save({
    'state_dict': final_mlp['model'].state_dict(),
    'hidden_sizes': list(best_mlp_cfg['hidden_sizes']),
    'dropout': float(best_mlp_cfg['dropout']),
    'in_dim': X_train_s.shape[1],
    'tuned_threshold': float(mlp_thr),
}, MODEL_DIR / 'best_mlp.pt')

# Save SVM (full pipeline; already fitted on X_train).
joblib.dump(final_svm, MODEL_DIR / 'best_svm.joblib')

# Config + expected test metrics for the test.ipynb PASS/FAIL check.
config = {
    'seed': SEED,
    'feature_cols': feature_cols,
    'mlp_config': {
        'hidden_sizes': list(best_mlp_cfg['hidden_sizes']),
        'dropout': float(best_mlp_cfg['dropout']),
        'lr': float(best_mlp_cfg['lr']),
        'tuned_threshold': float(mlp_thr),
        'imbalance_strategy': best_strategy['MLP'],
    },
    'svm_config': {
        'params': {k: (v if not isinstance(v, float) else float(v))
                   for k, v in svm_params.items()},
        'tuned_threshold': float(svm_thr),
        'imbalance_strategy': best_strategy['SVM'],
    },
    'expected_metrics': {
        'mlp_default': evaluate(y_test, mlp_test_proba, 0.5),
        'mlp_tuned':   evaluate(y_test, mlp_test_proba, mlp_thr),
        'svm_default': evaluate(y_test, svm_test_proba, 0.5),
        'svm_tuned':   evaluate(y_test, svm_test_proba, svm_thr),
    },
}
with open(MODEL_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Saved artefacts:")
for p in sorted(MODEL_DIR.glob('*')):
    if p.name.startswith('.'):
        continue
    print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size/1024:.1f} KB)")

Saved artefacts:
  models/best_mlp.pt  (50.8 KB)
  models/best_svm.joblib  (36.5 KB)
  models/config.json  (1.5 KB)
  models/scaler.joblib  (1.3 KB)


In [32]:
# --- Reload sanity check -----------------------------------------------------
sc_reload = joblib.load(MODEL_DIR / 'scaler.joblib')
svm_reload = joblib.load(MODEL_DIR / 'best_svm.joblib')
ckpt = torch.load(MODEL_DIR / 'best_mlp.pt', map_location=DEVICE)
mlp_reload = MLP(in_dim=ckpt['in_dim'],
                 hidden_sizes=tuple(ckpt['hidden_sizes']),
                 dropout=ckpt['dropout']).to(DEVICE)
mlp_reload.load_state_dict(ckpt['state_dict'])
mlp_reload.eval()

X_test_s_reload = sc_reload.transform(X_test)
mlp_proba_reload = mlp_reload.predict_proba(
    torch.tensor(X_test_s_reload, dtype=torch.float32, device=DEVICE)
).cpu().numpy()
svm_proba_reload = svm_reload.predict_proba(X_test)[:, 1]

print("Reload vs in-memory (max abs diff):")
print(f"  MLP: {np.max(np.abs(mlp_proba_reload - mlp_test_proba)):.2e}")
print(f"  SVM: {np.max(np.abs(svm_proba_reload - svm_test_proba)):.2e}")

reload_metrics = {
    'mlp_tuned': evaluate(y_test, mlp_proba_reload, ckpt['tuned_threshold']),
    'svm_tuned': evaluate(y_test, svm_proba_reload, svm_thr),
}
print("\nReload test metrics (MLP tuned):")
print(reload_metrics['mlp_tuned'])
print("Reload test metrics (SVM tuned):")
print(reload_metrics['svm_tuned'])

print("\n✓ Reload sanity check passed — test.ipynb will reproduce these exactly.")

Reload vs in-memory (max abs diff):
  MLP: 0.00e+00
  SVM: 0.00e+00

Reload test metrics (MLP tuned):
{'roc_auc': 0.9825954369753899, 'pr_auc': 0.9149484365458341, 'f1': 0.8795811518324608, 'precision': 0.9032258064516129, 'recall': 0.8571428571428571}
Reload test metrics (SVM tuned):
{'roc_auc': 0.9798723087435873, 'pr_auc': 0.9079261158407005, 'f1': 0.8235294117647058, 'precision': 0.9722222222222222, 'recall': 0.7142857142857143}

✓ Reload sanity check passed — test.ipynb will reproduce these exactly.


### Done.

Commit `outputs/`, `models/`, and `creditcard_reduced.csv` (or add them to
`.gitignore` if you prefer to keep the repo small), push, and then run
`test.ipynb` to verify reproducibility on a fresh clone.

Budget check: the notebook targets **under 10 minutes on CPU**. If your HPC run
exceeds that, check the `SLURM_CPUS_PER_TASK` allocation and consider
submitting to the GPU partition via `sbatch scripts/hpc/run_mlp.slurm`.